# Clusterização K-Means (Abordagem Pivotada de Gêneros Principais)

Neste notebook, aplicamos o algoritmo **K-Means** para agrupar os livros do Booklog em grandes "Tribos Literárias". 
Diferente da primeira tentativa (que utilizou redução de dimensionalidade linear via SVD), agora usaremos a **Abordagem B (Pivoted)**:
1. Trabalharemos no nível do livro (**84.054 registros**).
2. Utilizaremos **9 colunas binárias** correspondentes aos Gêneros Principais mapeados semânticamente na etapa anterior.
3. Utilizaremos as variáveis numéricas escalonadas (`rating`, `pages`, `totalratings`).
4. Treinaremos o modelo final com **K=3** para manter total compatibilidade com o layout e filtros do Dashboard atual.

Esta abordagem garante **100% de interpretabilidade**, pois os grupos serão definidos por categorias de gênero reais (como Romance, Fantasia) em vez de componentes matemáticos abstratos.


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import TruncatedSVD

SEED = 42
np.random.seed(SEED)
print("Bibliotecas importadas e sementes fixadas!")


Bibliotecas importadas e sementes fixadas!


## 1. Carregamento dos Dados
Carregamos o dataset de gêneros pivotados obtido na etapa de pré-processamento.


In [2]:
# Carregar os dados pivotados
df = pd.read_parquet('../data/processed/books_pivot_mapped.parquet')
print(f"Shape do dataset carregado: {df.shape}")
print(df.head())


Shape do dataset carregado: (81979, 14)
                                               title  ...  totalratings
0                                         "Daisuki."  ...          73.0
1                  "Dark Pictures" and Other Stories  ...          10.0
2             "Defects": Engendering the Modern Body  ...           4.0
3                              "Have-More" Plan, The  ...          70.0
4  "Headhunter" Hiring Secrets: The Rules of the ...  ...         146.0
[5 rows x 14 columns]


## 2. Preparação das Features e Escalonamento
Separamos os 9 gêneros principais das 3 variáveis numéricas. Em seguida, normalizamos as variáveis numéricas com `MinMaxScaler` para que todas as colunas fiquem em uma escala semelhante (de 0 a 1) antes de aplicar a métrica de distância euclidiana do K-Means.


In [3]:
# Colunas que serão usadas na modelagem
genre_cols = [
    'Artes, Lazer e Estilo de Vida',
    'Fantasia e Ficção Científica',
    'Ficção Geral e Literatura',
    'História e Biografia',
    'Infantojuvenil e Quadrinhos',
    'Mistério, Thriller e Terror',
    'Não-Ficção e Autodesenvolvimento',
    'Outros',
    'Romance'
]
num_cols = ['rating', 'pages', 'totalratings']

# Normalizar colunas numéricas
scaler = MinMaxScaler()
X_num = scaler.fit_transform(df[num_cols])

# Concatenar com as colunas binárias de gênero
X_genre = df[genre_cols].values
X_model = np.hstack((X_genre, X_num))

print(f"Shape da matriz de entrada para o K-Means: {X_model.shape}")
print("Exemplo do primeiro registro formatado:")
print(X_model[0])


Shape da matriz de entrada para o K-Means: (81979, 12)
Exemplo do primeiro registro formatado:
[0.00000000e+00 0.00000000e+00 1.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 1.00000000e+00 1.00000000e+00
 1.00000000e+00 7.54000000e-01 2.42857143e-03 1.91133200e-05]


## 3. Busca de Hiperparâmetros (Avaliação de K)
Rodamos o K-Means variando o número de clusters $K$ de 2 até 8. Avaliaremos a inércia (método do Cotovelo) e o Silhouette Score para entender o desempenho estatístico do agrupamento.


In [4]:
K_range = range(2, 9)
inercia_list = []
silhouette_list = []

# Amostra aleatória para o cálculo rápido do Silhouette Score (método pesado para datasets grandes)
sample_size = min(10000, X_model.shape[0])
sample_indices = np.random.choice(X_model.shape[0], sample_size, replace=False)
X_sample = X_model[sample_indices]

print("Executando avaliação para cada K...")
for k in K_range:
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=SEED, n_init=10)
    kmeans.fit(X_model)
    inercia_list.append(kmeans.inertia_)
    
    score = silhouette_score(X_sample, kmeans.labels_[sample_indices])
    silhouette_list.append(score)
    print(f"K={k} -> Inércia: {kmeans.inertia_:.0f} | Silhouette Score: {score:.4f}")


Executando avaliação para cada K...
K=2 -> Inércia: 116898 | Silhouette Score: 0.2152
K=3 -> Inércia: 102728 | Silhouette Score: 0.1884
K=4 -> Inércia: 94593 | Silhouette Score: 0.1940
K=5 -> Inércia: 87487 | Silhouette Score: 0.1898
K=6 -> Inércia: 82925 | Silhouette Score: 0.2027
K=7 -> Inércia: 77572 | Silhouette Score: 0.2093
K=8 -> Inércia: 74686 | Silhouette Score: 0.2097


### Visualização Gráfica da Busca de Hiperparâmetros
Plotamos os gráficos de Cotovelo e Silhouette Score para diagnóstico visual.


In [5]:
fig_eval = make_subplots(rows=1, cols=2, subplot_titles=("Método do Cotovelo (Inércia)", "Qualidade do Agrupamento (Silhouette)"))

# Inércia
fig_eval.add_trace(
    go.Scatter(x=list(K_range), y=inercia_list, mode='lines+markers', name='Inércia', marker_color='indigo'),
    row=1, col=1
)

# Silhouette
fig_eval.add_trace(
    go.Scatter(x=list(K_range), y=silhouette_list, mode='lines+markers', name='Silhouette', marker_color='plum'),
    row=1, col=2
)

fig_eval.update_layout(title_text="Métricas de Avaliação do K-Means (Abordagem Pivotada)", height=450, showlegend=False)
fig_eval.update_xaxes(title_text="Número de Clusters (K)", row=1, col=1)
fig_eval.update_xaxes(title_text="Número de Clusters (K)", row=1, col=2)
fig_eval.update_yaxes(title_text="Inércia", row=1, col=1)
fig_eval.update_yaxes(title_text="Silhouette Score", row=1, col=2)
fig_eval.show()


## 4. Treinamento do Modelo Final (K=4)
Selecionamos **K=4** para treinar o nosso modelo definitivo de Tribos Literárias.
Esta escolha baseia-se na melhor coesão estatística obtida no Elbow/Silhouette (Silhouette Score de **0.1940** comparado a 0.1884 de K=3) e na riqueza de segmentação, separando claramente as obras de Fantasia/Sci-Fi (Tribo Geek) dos Romances e Dramas Populares, que antes estavam misturados em um único grupo pop.


In [6]:
# Treinar o K-Means definitivo
K_FINAL = 4
kmeans_final = KMeans(n_clusters=K_FINAL, init='k-means++', random_state=SEED, n_init=10)
df['cluster'] = kmeans_final.fit_predict(X_model)

print("Distribuição de livros por cluster:")
print(df['cluster'].value_counts())


Distribuição de livros por cluster:
cluster
2    29162
1    19603
3    18014
0    15200
Name: count, dtype: int64


## 5. Perfilamento das Novas "Tribos Literárias"
Analisamos as características estatísticas médias (páginas, nota, quantidade de avaliações) e a presença de cada um dos 9 gêneros principais dentro de cada cluster para descrevê-los de forma semântica.


In [7]:
print("=== Perfis Numéricos das Novas Tribos ===")
perfil_num = df.groupby('cluster')[num_cols].mean()
print(perfil_num)

print("\n=== Taxa de Presença de Gêneros por Tribo (Valores de 0 a 1) ===")
perfil_genre = df.groupby('cluster')[genre_cols].mean()
print(perfil_genre.T.round(3))


=== Perfis Numéricos das Novas Tribos ===
           rating       pages  totalratings
cluster                                    
0        3.871078  246.158811   7973.572562
1        3.905489  290.469854   3638.436597
2        3.939912  279.840809    660.421633
3        3.817112  231.037226   3904.689251
=== Taxa de Presença de Gêneros por Tribo (Valores de 0 a 1) ===
cluster                               0      1      2      3
Artes, Lazer e Estilo de Vida     0.251  0.240  0.367  0.199
Fantasia e Ficção Científica      1.000  0.073  0.046  0.000
Ficção Geral e Literatura         0.868  1.000  0.000  0.775
História e Biografia              0.157  0.563  0.315  0.294
Infantojuvenil e Quadrinhos       0.461  0.211  0.072  0.262
Mistério, Thriller e Terror       0.306  0.114  0.023  0.221
Não-Ficção e Autodesenvolvimento  0.102  1.000  0.882  0.012
Outros                            0.599  0.824  0.749  0.429
Romance                           0.389  0.166  0.016  0.456


### Descrição das Novas Tribos Literárias (Storytelling)

Analisando a concentração de características e gêneros dominantes, descrevemos as 4 novas Tribos Literárias do Booklog:

1. **Tribo 0: "Universo Geek e Fantasia Pop"** (Composto 100% por Fantasia e Ficção Científica, com forte presença de Ficção Geral (87%) e Infantojuvenil (46%). É o grupo de maior apelo viral da plataforma, acumulando em média **7.973 avaliações por livro**).
2. **Tribo 1: "Literatura Sênior, Ensaios e Biografias"** (Composto 100% por Ficção Geral e Literatura e 100% por Não-Ficção e Autodesenvolvimento, com forte presença de História e Biografia (56%). Representa obras densas, como ensaios filosóficos, memórias e biografias literárias de alto nível, com média de 290 páginas).
3. **Tribo 2: "Não-Ficção de Nicho e Estilo de Vida"** (Composto principalmente por Não-Ficção e Autodesenvolvimento (88%), com forte presença de Artes/Estilo de Vida (37%) e Outros (75%). Reúne guias práticos, livros técnicos e manuais de hobbies, caracterizados por notas muito altas (3.94) e baixo volume de avaliações gerais (nicho)).
4. **Tribo 3: "Romances Mainstream e Dramas"** (Composto por Romance (46%) e Ficção Geral (77%), sem nenhuma presença de Fantasia. Reúne a ficção romântica, chick-lit e dramas populares. São os livros mais curtos em média (231 páginas), com a menor nota média do acervo (3.81), mas com alto engajamento da comunidade (média de 3.904 avaliações)).


## 6. Visualização 2D das Tribos Literárias (SVD)
Para mapear a estrutura multidimensional das Tribos Literárias (que foi treinada em uma matriz de 12 dimensões) no plano cartesiano, aplicamos o método **SVD (Singular Value Decomposition)** com 2 componentes.
Como nossa matriz de features é estritamente não-negativa (gêneros binários de 0 ou 1 e numéricos escalonados de 0 a 1), o SVD preserva a escala original não-negativa para o primeiro componente (variando de 0 a 2+), proporcionando um gráfico idêntico em estilo e limites ao gerado no Notebook 04 original.


In [8]:
from sklearn.decomposition import TruncatedSVD

# Aplicar TruncatedSVD para projetar as 12 features em 2D
svd_2d = TruncatedSVD(n_components=2, random_state=SEED)
X_svd = svd_2d.fit_transform(X_model)

# Salvar no dataframe original para exportação
df['svd_x'] = X_svd[:, 0]
df['svd_y'] = X_svd[:, 1]

print(f"Variância explicada pelos 2 primeiros componentes do SVD: {svd_2d.explained_variance_ratio_.sum() * 100:.2f}%")

# Criar DataFrame temporário para plotagem interativa (amostra de 10000 livros para performance)
df_plot = df.copy()
df_plot['Cluster'] = df_plot['cluster'].astype(str)
df_plot_sample = df_plot.sample(10000, random_state=SEED)

# Gráfico de dispersão interativo com Plotly Express (estilo padrão claro do Notebook 04)
fig_clusters = px.scatter(
    df_plot_sample, 
    x='svd_x', 
    y='svd_y', 
    color='Cluster', 
    hover_name='title',
    hover_data=['author', 'rating', 'pages'],
    title='Mapa Interativo das 4 Tribos Literárias (Amostra de 10000 livros)',
    category_orders={'Cluster': ['0', '1', '2', '3']},
    color_discrete_map={'0': '#e41a1c', '1': '#377eb8', '2': '#4daf4a', '3': '#ff7f00'},
    labels={'svd_x': 'Componente Principal 1 (SVD)', 'svd_y': 'Componente Principal 2 (SVD)'}
)

fig_clusters.update_traces(marker=dict(size=5, opacity=0.6, line=dict(width=0.3, color='DarkSlateGrey')))
fig_clusters.show()


## 7. Exportação dos Dados com Clusters
Salvamos o dataset final com as novas atribuições de clusters para integração. Para não mexer no código do dashboard, salvaremos os novos resultados exatamente com o nome e caminho esperados: `livros_com_clusters.parquet` e `livros_com_clusters.csv`.


In [9]:
# Renomear 'cluster' para 'Cluster' para manter compatibilidade com o dashboard
df_export = df.rename(columns={'cluster': 'Cluster'})

# Salvar no diretório de processados, substituindo os antigos clusters
df_export.to_parquet('../data/processed/livros_com_clusters.parquet', index=False)
df_export.to_csv('../data/processed/livros_com_clusters.csv', index=False)
print("Novos clusters exportados com sucesso para livros_com_clusters.parquet e livros_com_clusters.csv!")


Novos clusters exportados com sucesso para livros_com_clusters.parquet e livros_com_clusters.csv!
